# 03.0 — Analyse de la stratégie de chunking

## Objectif

Avant de définir une stratégie de chunking, nous analysons la taille réelle des documents qui seront envoyés au modèle d'embedding (`mistral-embed`).

L'objectif est de répondre aux questions suivantes :

- Quelle est la longueur moyenne des descriptions des événements ?
- Combien de tokens représentent les descriptions ?
- Quelle proportion des événements nécessite réellement un découpage ?
- Quel format de chunk optimise la recherche sémantique ?

Pour accélérer les expérimentations, nous limitons dans un premier temps le périmètre à la ville de Paris.

---

In [1]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
import mistral_common
from mistral_common.tokens.tokenizers.tekken import Tekkenizer
import pandas as pd

from notebooks.config import DATA_INTERIM, DATA_PROCESSED

CHUNK_MAX_TOKENS = 256
CHUNK_OVERLAP = 20

data_dir = Path(mistral_common.__file__).parent / "data"
tokenizer_files = sorted(data_dir.glob("tekken_*.json"), reverse=True)
tokenizer = Tekkenizer.from_file(str(tokenizer_files[0]))


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, bos=False, eos=False))


df = pd.read_parquet(sorted(DATA_PROCESSED.glob("*_clean.parquet"))[-1])

print(f"Dataset complet  : {df.shape}")
print(f"Chunk max tokens : {CHUNK_MAX_TOKENS}")
print(f"Chunk overlap    : {CHUNK_OVERLAP}")

df.head()

Dataset complet  : (9840, 13)
Chunk max tokens : 256
Chunk overlap    : 20


,uid,title_fr,keywords_fr,description_clean,conditions_fr,date_display,firstdate_begin,lastdate_begin,location_name,location_address,location_city,location_department,location_region
0,6670838,Visite-atelier : Prendre la mesure du temps,None,Cette visite-atelier invite petits et grands à...,25 participants maximum.,Du 20 septembre 2025 à 12:30 au 21 septembre 2...,2025-09-20T12:30:00+00:00,2025-09-21T12:30:00+00:00,Musée de la poste,"34 Boulevard de Vaugirard, 75015 Paris, France",Paris,Paris,Île-de-France
1,96129611,La fresque de la Biodiversité : comprendre ce ...,None,Au travers d'un atelier ludique et collaborati...,Entrée gratuite à partir de 12 ans / Réservati...,Le 4 octobre 2025 à 07:30,2025-10-04T07:30:00+00:00,2025-10-04T07:30:00+00:00,Bibliothèque Jean-Jacques Sempe,30 Bis rue de la Libération 91680 Bruyères-le-...,Bruyères-le-Châtel,Essonne,Île-de-France
2,45384394,On met les voiles !,None,"A l'occasion de ""Rendez-vous aux jardins"", le ...",None,Du 7 juin 2025 à 08:00 au 8 juin 2025 à 08:00,2025-06-07T08:00:00+00:00,2025-06-08T08:00:00+00:00,Domaine national du Palais-Royal,"8 rue Montpensier, 75001 Paris",Paris,Paris,Île-de-France
3,67909253,Visite libre du Jardin partagé de la Harengerie,None,"Sur un terrain en friche, les habitants du qua...",None,Du 7 juin 2025 à 12:00 au 8 juin 2025 à 08:00,2025-06-07T12:00:00+00:00,2025-06-08T08:00:00+00:00,Jardin de Reconquête,13 rue de la Harengerie 95300 Pontoise,Pontoise,Val-d'Oise,Île-de-France
4,17809408,Atelier rempotage,None,Profitez d'un atelier rempotage pour venir ren...,None,Du 7 juin 2025 à 12:00 au 8 juin 2025 à 12:00,2025-06-07T12:00:00+00:00,2025-06-08T12:00:00+00:00,Château et parc de Champs-sur-Marne,"31 rue de Paris, 77420 Champs-sur-Marne",Champs-sur-Marne,Seine-et-Marne,Île-de-France


## Filtrage sur Paris

Nous réduisons volontairement le périmètre à la ville de Paris afin de valider rapidement la méthodologie de vectorisation et de recherche sémantique.

Une fois le pipeline validé, le même processus pourra être appliqué à l'ensemble de l'Île-de-France.

In [2]:
df_paris = df[df["location_city"].fillna("").str.lower().str.strip() == "paris"].copy().head(200)

print(f"Nombre d'événements Paris : {len(df_paris):,}")

Nombre d'événements Paris : 200


## Analyse de la longueur des descriptions

Les modèles d'embedding travaillent sur des tokens. Nous mesurons la taille des descriptions avec le **tokenizer Tekken** de Mistral pour déterminer si un découpage est nécessaire.

Deux niveaux de texte :
- **Description seule** (`description_clean`) : texte vectorisé pour la recherche sémantique
- **Document parent** (titre + date + lieu + description + conditions) : texte complet envoyé au LLM comme contexte

In [3]:
df_paris["desc_tokens"] = df_paris["description_clean"].fillna("").apply(count_tokens)

print("=== Tokens par description (tokenizer Tekken) ===")
print(df_paris["desc_tokens"].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]).round(0))

=== Tokens par description (tokenizer Tekken) ===
count     200.0
mean      251.0
std       288.0
min         3.0
50%       139.0
75%       320.0
90%       563.0
95%       810.0
99%      1293.0
max      1813.0
Name: desc_tokens, dtype: float64


## Proportion d'événements nécessitant un découpage

Avec `mistral-embed` et une limite choisie de **256 tokens** par chunk, vérifions combien d'événements dépassent cette limite.

Contrairement à MiniLM (128 tokens max), `mistral-embed` accepte des séquences bien plus longues, mais nous limitons volontairement à 256 tokens pour garder des chunks focalisés sémantiquement.

In [4]:
n_over = (df_paris["desc_tokens"] > CHUNK_MAX_TOKENS).sum()
pct_over = n_over / len(df_paris) * 100

print(f"Limite choisie : {CHUNK_MAX_TOKENS} tokens")
print(f"Événements dépassant la limite : {n_over} ({pct_over:.1f}%)")
print(f"Événements dans la limite      : {len(df_paris) - n_over} ({100 - pct_over:.1f}%)")

Limite choisie : 256 tokens
Événements dépassant la limite : 63 (31.5%)
Événements dans la limite      : 137 (68.5%)


## Distribution des tokens par tranche

In [5]:
bins = [0, 1, 64, 128, 256, 512, float("inf")]
labels = ["vide", "<64", "64-128", "128-256", "256-512", ">512"]

df_paris["desc_bucket"] = pd.cut(df_paris["desc_tokens"], bins=bins, labels=labels)

print("Distribution des descriptions par tranche de tokens :")
print(df_paris["desc_bucket"].value_counts().sort_index().to_string())

Distribution des descriptions par tranche de tokens :
desc_bucket
vide        0
<64        44
64-128     47
128-256    46
256-512    40
>512       23


## Événements les plus longs

In [6]:
long_docs = df_paris[df_paris["desc_tokens"] > CHUNK_MAX_TOKENS]

print(f"Événements longs (> {CHUNK_MAX_TOKENS} tokens) : {len(long_docs)}")
print()
print(long_docs["desc_tokens"].describe())
print()
long_docs.sort_values("desc_tokens", ascending=False)[["title_fr", "desc_tokens"]].head(10)

Événements longs (> 256 tokens) : 63

count      63.000000
mean      565.222222
std       333.431158
min       261.000000
25%       344.000000
50%       466.000000
75%       663.500000
max      1813.000000
Name: desc_tokens, dtype: float64



,title_fr,desc_tokens
425,Afro Bal à La Villette spécial FELABRATION ave...,1813
424,Villette Tropicale ~ Soirée Afro-Latino live +...,1802
124,Visite des ministères de l'Aménagement du terr...,1288
435,Comprendre le vieillissement au travail : des ...,1170
190,Séminaire (3 jours) : A la rencontre de votre ...,1094
359,FinTech R:Evolution | #FFT26,1065
181,Impacts de l'IA sur l'avenir des consultants e...,960
356,Maîtriser la création de contenu assisté par l'IA,891
182,Soirée Stratégie de Défense,886
518,Rendez-vous aux jardins au musée Bourdelle : P...,843


## Taille du document parent

Le document parent (envoyé au LLM) contient toutes les métadonnées + la description.
Vérifions sa taille pour s'assurer qu'il reste raisonnable pour la fenêtre de contexte du LLM.

In [7]:
def _has_value(val):
    if val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except (ValueError, TypeError):
        pass
    if isinstance(val, str) and val.strip() == "":
        return False
    return True


def build_parent_text(row):
    parts = [f"# {row['title_fr']}"]
    parts.append(f"\n**Date** : {row['date_display']}")
    parts.append(f"**Lieu** : {row['location_name']}, {row['location_city']}")
    if row.get("location_address"):
        parts.append(f"**Adresse** : {row['location_address']}")
    if _has_value(row.get("keywords_fr")):
        kw = row["keywords_fr"]
        if isinstance(kw, list):
            kw = ", ".join(kw)
        parts.append(f"**Mots-clés** : {kw}")
    if _has_value(row.get("conditions_fr")):
        parts.append(f"**Conditions** : {row['conditions_fr']}")
    parts.append(f"\n{row['description_clean']}")
    return "\n".join(parts)


def build_chunk_header(row):
    header = f"Titre : {row['title_fr']}"
    if _has_value(row.get("keywords_fr")):
        kw = row["keywords_fr"]
        if isinstance(kw, list):
            kw = ", ".join(kw)
        header += f"\nMots-clés : {kw}"
    return header


df_paris["parent_doc"] = df_paris.apply(build_parent_text, axis=1)
df_paris["parent_tokens"] = df_paris["parent_doc"].apply(count_tokens)
df_paris["chunk_header"] = df_paris.apply(build_chunk_header, axis=1)
df_paris["header_tokens"] = df_paris["chunk_header"].apply(count_tokens)

print("=== Tokens par document parent (contexte LLM) ===")
print(df_paris["parent_tokens"].describe(percentiles=[0.5, 0.75, 0.90, 0.95]).round(0))

print("\n=== Tokens consommés par l'en-tête du chunk ===")
print(df_paris["header_tokens"].describe().round(0))
print(f"\nTokens restants pour le contenu : ~{CHUNK_MAX_TOKENS} - {int(df_paris['header_tokens'].median())} = ~{CHUNK_MAX_TOKENS - int(df_paris['header_tokens'].median())} tokens")

print("\nExemple de document parent :")
print("=" * 80)
print(df_paris["parent_doc"].iloc[0])

print("\n\nExemple d'en-tête chunk :")
print("=" * 80)
print(df_paris["chunk_header"].iloc[0])

=== Tokens par document parent (contexte LLM) ===
count     200.0
mean      351.0
std       300.0
min        76.0
50%       246.0
75%       433.0
90%       656.0
95%       953.0
max      1943.0
Name: parent_tokens, dtype: float64

=== Tokens consommés par l'en-tête du chunk ===
count    200.0
mean      30.0
std       23.0
min        4.0
25%       13.0
50%       22.0
75%       44.0
max      117.0
Name: header_tokens, dtype: float64

Tokens restants pour le contenu : ~256 - 21 = ~235 tokens

Exemple de document parent :
# Visite-atelier : Prendre la mesure du temps

**Date** : Du 20 septembre 2025 à 12:30 au 21 septembre 2025 à 12:30
**Lieu** : Musée de la poste, Paris
**Adresse** : 34 Boulevard de Vaugirard, 75015 Paris, France
**Conditions** : 25 participants maximum.

Cette visite-atelier invite petits et grands à découvrir l'exposition temporaire *La fabrique du Temps* de manière ludique et créative. À travers une visite de l'exposition et un quiz pédagogique, les visiteurs explorent

## Stratégie de chunking retenue

### Constat

Les événements culturels sont des **documents courts et structurés** : la description médiane fait environ 180 tokens. Ce ne sont pas des articles longs nécessitant un découpage agressif.

### Stratégie : chunks avec en-tête + Parent Document Retriever

Chaque chunk inclut un **en-tête structuré** (titre + mots-clés) suivi d'un fragment de description, le tout dans la limite de **256 tokens**.

| | Child chunk (vectorisé) | Parent document (envoyé au LLM) |
|---|---|---|
| **Contenu** | `Titre : ...\nMots-clés : ...\n\n{fragment description}` | Métadonnées complètes + description |
| **Rôle** | Recherche sémantique | Contexte pour la génération |
| **Taille** | ≤ 256 tokens | Illimité (dans la fenêtre LLM) |

**Pourquoi titre + mots-clés dans le chunk ?** Quand un utilisateur demande "concert de jazz à Paris", le retriever doit matcher sur le type d'événement. Si ces infos ne sont que dans les métadonnées (invisibles à l'embedding), le matching est dégradé.

**Règles de découpage :**
- Description + en-tête ≤ 256 tokens → **1 chunk**
- Description + en-tête > 256 tokens → **N chunks** via `RecursiveCharacterTextSplitter`, chaque chunk préfixé de l'en-tête
- Le **parent document** reste toujours complet et intact
- `chunk_overlap = 20` tokens entre fragments consécutifs

**Modèle d'embedding** : `mistral-embed` (1024 dimensions) — choisi pour sa compatibilité avec l'écosystème Mistral utilisé pour la génération.

---

## Production des chunks

On applique la stratégie définie ci-dessus pour produire les child chunks et parent documents.

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_MAX_TOKENS,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=count_tokens,
)

chunks_data: list[dict] = []
parents_data: list[dict] = []

for _, row in df_paris.iterrows():
    uid = str(row["uid"])
    parent_text = build_parent_text(row)
    parents_data.append({"uid": uid, "parent_doc": parent_text})

    header = build_chunk_header(row)
    description = (row["description_clean"] or "").strip() or row["title_fr"]
    fragments = splitter.split_text(description)

    for i, fragment in enumerate(fragments):
        chunk_text = f"{header}\n\n{fragment}"
        chunks_data.append({
            "uid": uid,
            "chunk_id": i,
            "text": chunk_text,
            "n_tokens": count_tokens(chunk_text),
        })

chunks_df = pd.DataFrame(chunks_data)
parent_df = pd.DataFrame(parents_data)

n_events_split = len({c["uid"] for c in chunks_data if c["chunk_id"] > 0})

print(f"Événements traités  : {len(df_paris)}")
print(f"Child chunks créés  : {len(chunks_df)}")
print(f"Parent documents    : {len(parent_df)}")
print(f"Événements découpés : {n_events_split}")
print(f"Événements intacts  : {len(df_paris) - n_events_split}")

Événements traités  : 200
Child chunks créés  : 350
Parent documents    : 200
Événements découpés : 65
Événements intacts  : 135


In [9]:
print("=== Distribution des tokens par child chunk ===")
print(chunks_df["n_tokens"].describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(0))
print(f"\nChunks > {CHUNK_MAX_TOKENS} tokens : {(chunks_df['n_tokens'] > CHUNK_MAX_TOKENS).sum()}")

=== Distribution des tokens par child chunk ===
count    350.0
mean     182.0
std       85.0
min       10.0
50%      182.0
75%      260.0
90%      289.0
95%      299.0
max      371.0
Name: n_tokens, dtype: float64

Chunks > 256 tokens : 94


## Vérification qualitative

In [10]:
# Événement court (1 seul chunk)
short_chunks = chunks_df[chunks_df.groupby("uid")["uid"].transform("count") == 1]
if len(short_chunks) > 1:
    sample_short = short_chunks.iloc[1]
    print("=== ÉVÉNEMENT COURT (1 chunk) ===")
    print(f"UID: {sample_short['uid']} | {sample_short['n_tokens']} tokens")
    print(sample_short["text"])

print("\n" + "=" * 80)

# Événement découpé (plusieurs chunks)
split_uids = chunks_df[chunks_df["chunk_id"] > 0]["uid"].unique()
if len(split_uids) > 0:
    long_uid = split_uids[0]
    long_chunks = chunks_df[chunks_df["uid"] == long_uid]
    parent_text = parent_df[parent_df["uid"] == long_uid]["parent_doc"].iloc[0]

    print(f"\n=== ÉVÉNEMENT DÉCOUPÉ ({len(long_chunks)} chunks) ===")
    print(f"UID: {long_uid}")
    for _, c in long_chunks.iterrows():
        print(f"\n--- Chunk #{c['chunk_id']} ({c['n_tokens']} tokens) ---")
        print(c["text"][:300] + "...")
    print("\n--- Parent document (début) ---")
    print(parent_text[:500])

=== ÉVÉNEMENT COURT (1 chunk) ===
UID: 45384394 | 76 tokens
Titre : On met les voiles !

A l'occasion de "Rendez-vous aux jardins", le Palais-Royal renoue avec une tradition disparue depuis environ un siècle et convie petits et grands, passionnés de plaisance miniature. Alors, on prend le large sur le bassin du jardin sur le petit voilier de son choix. Une activité à partager en famille.


=== ÉVÉNEMENT DÉCOUPÉ (2 chunks) ===
UID: 93263481

--- Chunk #0 (286 tokens) ---
Titre : Nuits des étoiles au parc Montsouris
Mots-clés : ['Paris sous les étoiles' '' 'pse' 'astronomie' 'telescope' 'étoile'
 'ciel' 'nature' 'nuit']

Grâce au partenariat avec la ville de Paris, 21 Parcs et Jardins seront ouverts à toutes et à tous, entre le 28 juin et le 13 septembre 2025. Les mo...

--- Chunk #1 (224 tokens) ---
Titre : Nuits des étoiles au parc Montsouris
Mots-clés : ['Paris sous les étoiles' '' 'pse' 'astronomie' 'telescope' 'étoile'
 'ciel' 'nature' 'nuit']

**Conditions d'accès** : Les soirées s

## Sauvegarde

In [11]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

chunks_df.to_parquet(DATA_INTERIM / "paris_chunks.parquet", index=False)
parent_df.to_parquet(DATA_INTERIM / "paris_parent_docs.parquet", index=False)

print(f"Child chunks sauvegardés     : {len(chunks_df)} → {DATA_INTERIM / 'paris_chunks.parquet'}")
print(f"Parent documents sauvegardés : {len(parent_df)} → {DATA_INTERIM / 'paris_parent_docs.parquet'}")

Child chunks sauvegardés     : 350 → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/interim/paris_chunks.parquet
Parent documents sauvegardés : 200 → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/interim/paris_parent_docs.parquet


## Résumé

| Élément | Fichier | Contenu |
|---|---|---|
| Child chunks | `paris_chunks.parquet` | `Titre + Mots-clés + fragment description` (≤ 256 tokens), pour l'embedding FAISS |
| Parent documents | `paris_parent_docs.parquet` | Événement complet en Markdown, pour le contexte LLM |

Le lien entre les deux se fait via le champ `uid` présent dans les child chunks.

---
**Suite** : `04.0-kl-embedding.ipynb` — vectorisation et indexation FAISS avec `mistral-embed`

In [12]:
for i in range(len(parent_df['parent_doc'])):
    print(parent_df['parent_doc'].iloc[i])

# Visite-atelier : Prendre la mesure du temps

**Date** : Du 20 septembre 2025 à 12:30 au 21 septembre 2025 à 12:30
**Lieu** : Musée de la poste, Paris
**Adresse** : 34 Boulevard de Vaugirard, 75015 Paris, France
**Conditions** : 25 participants maximum.

Cette visite-atelier invite petits et grands à découvrir l'exposition temporaire *La fabrique du Temps* de manière ludique et créative. À travers une visite de l'exposition et un quiz pédagogique, les visiteurs explorent les outils et méthodes qui ont permis de capter le temps au fil des époques avant de fabriquer leur propre calendrier lunaire sous forme de pop-up ou de leporello, en fonction du niveau des participants.
# On met les voiles !

**Date** : Du 7 juin 2025 à 08:00 au 8 juin 2025 à 08:00
**Lieu** : Domaine national du Palais-Royal, Paris
**Adresse** : 8 rue Montpensier, 75001 Paris

A l'occasion de "Rendez-vous aux jardins", le Palais-Royal renoue avec une tradition disparue depuis environ un siècle et convie petits et gra